# Chapter 8: Calculation of Molecular Properties
## 8.4. Potential energy surfaces

How does a molecule's energy change when we stretch a bond or bend an angle? We will construct a small **ethanol MMFF94 energy surface** over the C–O bond length and C–C–O angle. Every point starts from the same molecular conformation, and all calculations finish quickly on an ordinary CPU.

### Learning objectives

By the end, you should be able to:

- define a potential energy surface and explain what a two-dimensional slice leaves out;
- construct a reproducible scan with explicit atom indices, coordinate units, and a common energy zero;
- distinguish an unrelaxed scan, a constrained relaxation, and a free-energy surface;
- interpret contours and one-dimensional cuts without identifying a grid minimum as a proven stationary point;
- state why a conventional molecular mechanics model cannot predict bond dissociation or an electronic excitation.

**Setup:** use the course environment in the [README](Readme.md). This notebook is independent of the other parts. It uses RDKit, NumPy, pandas, and Matplotlib, with no electronic-structure jobs, downloads, viewer dependencies, or prerequisite output files. The grid has 169 inexpensive force-field evaluations; the small relaxation example has seven bounded optimizations.

### Start here: a molecular energy map

Picture changing one bond length while holding a molecular structure in your hands. Stretching far enough usually costs energy. Changing a bond angle can also cost energy, and the two motions may influence each other. A **potential-energy surface** records those energy changes as a function of nuclear coordinates.

- A slope (first derivative) gives a force with the opposite sign.
- Curvature (second derivative) tells us how rapidly that force changes near a point.
- A minimum has no downhill internal direction locally; a two-coordinate slice can hide other directions.

Here the inexpensive MMFF94 force field supplies the energy. The quantum Born–Oppenheimer surface is the conceptual connection, not the engine used for the grid. **First pass:** read the coordinate picture, contours, and relaxed/unrelaxed comparison. The new quadratic-fit test asks a practical question that leads directly to the harmonic vibration model in Part 5.

In [ ]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem, rdMolTransforms

output_dir = Path("outputs/chapter08_part4")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"RDKit {rdBase.rdkitVersion}; NumPy {np.__version__}")

### 8.4.1. Which surface are we calculating?

Within the Born–Oppenheimer approximation, an electronic potential energy surface for one electronic state is

$$
E_\mathrm{BO}(\mathbf R)=E_\mathrm{electronic}(\mathbf R)+V_\mathrm{nuclear\ repulsion}(\mathbf R),
$$
where $\mathbf R$ gives the nuclear positions. A nonlinear molecule with $N$ atoms has $3N-6$ internal degrees of freedom. Ethanol has nine atoms including hydrogen, so its internal configuration requires **21 coordinates**, not just the two drawn here.

This notebook evaluates $E_\mathrm{MMFF94}(\mathbf R)$, a parameterized molecular-mechanics approximation to a ground-state potential. It includes bonded and nonbonded terms for a fixed molecular graph. It is useful near ordinary organic-molecule geometries, but is not an electronic-structure calculation. Its absolute energy zero is a force-field convention; it cannot be compared directly to an HF/DFT total energy or a MOPAC heat of formation.

We choose

$$
q_1=r_{\mathrm{C-O}}\quad\text{in Å},\qquad q_2=\theta_{\mathrm{C-C-O}}\quad\text{in degrees}.
$$

A scan must also specify **how all the other coordinates are assigned**. Merely setting these two values in newly embedded random conformers would mix conformational differences into the apparent surface.

### 8.4.2. Prepare one reference geometry

In the SMILES `CCO`, the heavy atoms have indices **C0–C1–O2**. Adding explicit hydrogens keeps those indices and appends the six H atoms. A seeded ETKDG embedding is followed by MMFF94 optimization. We check that parameters exist and that the optimizer converges.

This produces a reproducible **optimized reference geometry**, not a proof that every ethanol conformer has been explored. Fixing the seed stabilizes the workflow; small numerical differences may remain between software versions.

In [ ]:
reference = Chem.AddHs(Chem.MolFromSmiles("CCO"))
assert [reference.GetAtomWithIdx(i).GetSymbol() for i in (0, 1, 2)] == ["C", "C", "O"]
assert reference.GetNumAtoms() == 9
embedding = AllChem.ETKDGv3()
embedding.randomSeed = 20260804
embedding.numThreads = 1
if AllChem.EmbedMolecule(reference, embedding) != 0:
    raise RuntimeError("Ethanol embedding failed.")
if not AllChem.MMFFHasAllMoleculeParams(reference):
    raise RuntimeError("MMFF94 parameters are missing.")


def mmff_force_field(molecule):
    properties = AllChem.MMFFGetMoleculeProperties(molecule, mmffVariant="MMFF94")
    if properties is None:
        raise RuntimeError("Could not assign MMFF94 properties.")
    force_field = AllChem.MMFFGetMoleculeForceField(molecule, properties, confId=0)
    if force_field is None:
        raise RuntimeError("Could not create the force field.")
    return force_field


reference_ff = mmff_force_field(reference)
status = reference_ff.Minimize(maxIts=500, forceTol=1e-5, energyTol=1e-9)
if status != 0:
    raise RuntimeError(f"Reference minimization did not converge (status {status}).")
reference_energy = float(reference_ff.CalcEnergy())
reference_xyz = reference.GetConformer().GetPositions().copy()
r0 = rdMolTransforms.GetBondLength(reference.GetConformer(), 1, 2)
theta0 = rdMolTransforms.GetAngleDeg(reference.GetConformer(), 0, 1, 2)
print(f"MMFF94 reference energy: {reference_energy:.6f} kcal/mol")
print(f"C1–O2: {r0:.6f} Å; C0–C1–O2: {theta0:.6f}°; optimizer status: {status}")

In [ ]:
# Show the heavy-atom geometry in its own plane, using the same indexed atoms.
heavy_xyz = reference_xyz[:3]
origin = heavy_xyz[1]
u = (heavy_xyz[0] - origin) / np.linalg.norm(heavy_xyz[0] - origin)
v = heavy_xyz[2] - origin
v = v - np.dot(v, u) * u
v /= np.linalg.norm(v)
projected = np.column_stack([(heavy_xyz - origin) @ u, (heavy_xyz - origin) @ v])
fig, ax = plt.subplots(figsize=(5.5, 3.6), layout="constrained")
ax.plot(projected[:, 0], projected[:, 1], color="0.55", linewidth=3, zorder=1)
ax.scatter(projected[:, 0], projected[:, 1], c=["#555555", "#555555", "#b43e50"], s=650, zorder=2)
for label, (px, py) in zip(["C0", "C1", "O2"], projected):
    ax.text(px, py, label, color="white", ha="center", va="center", weight="bold")
ax.set(aspect="equal", title="Reference heavy-atom plane\nHydrogens are present in the calculation",
       xlabel="In-plane coordinate (Å)", ylabel="In-plane coordinate (Å)")
ax.margins(0.25)
plt.show()

### 8.4.3. Define the deformation before evaluating its energy

At each grid point we **copy the same reference** and then set the C–O distance and C–C–O angle, in that order. RDKit moves the connected O–H group with the oxygen. This preserves that group's internal geometry while changing its position relative to the carbon framework. The carbon framework and its attached hydrogens remain fixed.

We call this an **unrelaxed scan along a specified deformation**: nothing is energy-minimized after the setters run. It is not a claim that every other redundant bond angle in the molecule remains numerically unchanged. For example, moving O also changes its angles to the hydrogens attached to C1.

This definition makes each geometry a deterministic function $\mathbf R_\mathrm{scan}(r,\theta)$. Both requested coordinates are measured again after setting them. The grid stays close to the reference (±0.16 Å and ±16°); stretching a fixed-topology force field far toward dissociation would not model chemical bond breaking.

In [ ]:
def scan_geometry(bond_length, bond_angle):
    molecule = Chem.Mol(reference)
    conformer = molecule.GetConformer()
    rdMolTransforms.SetBondLength(conformer, 1, 2, float(bond_length))
    rdMolTransforms.SetAngleDeg(conformer, 0, 1, 2, float(bond_angle))
    np.testing.assert_allclose(rdMolTransforms.GetBondLength(conformer, 1, 2), bond_length, atol=1e-10)
    np.testing.assert_allclose(rdMolTransforms.GetAngleDeg(conformer, 0, 1, 2), bond_angle, atol=1e-9)
    return molecule


# A deformation of zero must recover the original coordinates and energy.
recovered = scan_geometry(r0, theta0)
np.testing.assert_allclose(recovered.GetConformer().GetPositions(), reference_xyz, atol=1e-9)
np.testing.assert_allclose(mmff_force_field(recovered).CalcEnergy(), reference_energy, atol=1e-8)
print("Reference geometry and energy are recovered at the central grid point.")

In [ ]:
bond_lengths = r0 + np.linspace(-0.16, 0.16, 13)
bond_angles = theta0 + np.linspace(-16, 16, 13)
# Axis 0 indexes lengths; axis 1 indexes angles.
energy_grid = np.empty((len(bond_lengths), len(bond_angles)))
start = perf_counter()
for i, bond_length in enumerate(bond_lengths):
    for j, bond_angle in enumerate(bond_angles):
        molecule = scan_geometry(bond_length, bond_angle)
        energy_grid[i, j] = mmff_force_field(molecule).CalcEnergy()
scan_seconds = perf_counter() - start
assert np.all(np.isfinite(energy_grid))
np.testing.assert_allclose(reference.GetConformer().GetPositions(), reference_xyz, atol=1e-12)
relative_grid = energy_grid - reference_energy  # one shared energy zero throughout
center_i, center_j = len(bond_lengths) // 2, len(bond_angles) // 2
np.testing.assert_allclose(relative_grid[center_i, center_j], 0, atol=1e-8)
minimum_i, minimum_j = np.unravel_index(np.argmin(energy_grid), energy_grid.shape)
print(f"{energy_grid.size} MMFF94 evaluations: {scan_seconds:.3f} s")
print(f"Lowest sampled point: r = {bond_lengths[minimum_i]:.4f} Å, θ = {bond_angles[minimum_j]:.3f}°")
print(f"Sampled relative energies: {relative_grid.min():.4f} to {relative_grid.max():.4f} kcal/mol")

### 8.4.4. Contour map and surface

Both plots show $\Delta E=E_\mathrm{MMFF94}(r,\theta)-E_\mathrm{reference}$ in kcal/mol. `meshgrid(..., indexing="ij")` matches the energy array's length-first, angle-second indexing; no unexplained transpose is needed.

A contour connects points with equal energy. Closer contours indicate a steeper change with position on the displayed axes, whose units differ. The connecting surface is only a visualization of sampled values. Neither interpolation nor the lowest grid entry proves the position or character of a stationary point in all 21 internal coordinates.

In [ ]:
R, Theta = np.meshgrid(bond_lengths, bond_angles, indexing="ij")
assert R.shape == Theta.shape == relative_grid.shape
fig = plt.figure(figsize=(11.5, 4.6), layout="constrained")
ax_map = fig.add_subplot(1, 2, 1)
contours = ax_map.contourf(R, Theta, relative_grid, levels=16, cmap="viridis")
ax_map.contour(R, Theta, relative_grid, levels=8, colors="white", linewidths=0.5, alpha=0.7)
ax_map.scatter([r0], [theta0], marker="x", s=80, color="#df4b36", linewidth=2, label="reference")
ax_map.set(xlabel="C–O length (Å)", ylabel="C–C–O angle (°)", title="Unrelaxed MMFF94 contour map")
ax_map.legend(loc="upper right")
fig.colorbar(contours, ax=ax_map, label="ΔE (kcal/mol)", shrink=0.85)
ax_surface = fig.add_subplot(1, 2, 2, projection="3d")
ax_surface.plot_surface(R, Theta, relative_grid, cmap="viridis", linewidth=0, antialiased=True)
ax_surface.set(xlabel="C–O length (Å)", ylabel="C–C–O angle (°)", title="Same sampled energies", zlabel="")
ax_surface.text2D(1.04, 0.5, "ΔE (kcal/mol)", transform=ax_surface.transAxes,
                  rotation=90, va="center")
ax_surface.view_init(elev=28, azim=-130)
plt.show()

### 8.4.5. Read one-dimensional cuts carefully

A cut at a **fixed C–O length** follows angle values along one row of the energy array. A cut at a **fixed angle** follows a column of the array as length changes. Every curve below uses the same reference energy; subtracting each curve's own minimum would hide the energy cost of the imposed constraint.

These shapes include coupling between coordinates, from explicit force-field cross terms and from the geometry dependence of other bonded/nonbonded contributions. A quadratic approximation can be useful very near a stable structure, but symmetry or perfect parabolas are not required.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for i in [2, center_i, len(bond_lengths) - 3]:
    axes[0].plot(bond_angles, relative_grid[i, :], "o-", markersize=3,
                 label=f"r = {bond_lengths[i]:.3f} Å")
axes[0].set(xlabel="C–C–O angle (°)", ylabel="ΔE (kcal/mol)", title="Angle cuts at fixed C–O length")
axes[0].legend()
for j in [2, center_j, len(bond_angles) - 3]:
    axes[1].plot(bond_lengths, relative_grid[:, j], "o-", markersize=3,
                 label=f"θ = {bond_angles[j]:.1f}°")
axes[1].set(xlabel="C–O length (Å)", ylabel="ΔE (kcal/mol)", title="Length cuts at fixed C–C–O angle")
axes[1].legend()
plt.show()

### Worked research decision: how far can a local harmonic picture be trusted?

A harmonic model replaces an energy surface near a minimum by a quadratic “bowl.” Fit that bowl using **only the central 3 × 3 points** of the existing grid, then compare its extrapolation with the full MMFF94 slice. This adds no force-field calculations.

To keep unlike coordinate units explicit, use dimensionless $x=(r-r_0)/(0.1\ \mathrm{Å})$ and $y=(\theta-\theta_0)/(10^\circ)$. Fit $E_{\rm quad}=c+g_x x+g_y y+\tfrac12 h_{xx}x^2+h_{xy}xy+\tfrac12h_{yy}y^2$. The cross term allows the two deformations to couple. Coefficients refer to these scaled coordinates, not to physical vibrational frequencies.

In [ ]:
scaled_r = (bond_lengths - r0) / 0.1
scaled_theta = (bond_angles - theta0) / 10.0
qx, qy = np.meshgrid(scaled_r, scaled_theta, indexing="ij")
quadratic_design = np.stack([np.ones_like(qx), qx, qy, 0.5*qx**2, qx*qy, 0.5*qy**2], axis=-1)
fit_mask = np.zeros(relative_grid.shape, dtype=bool)
fit_mask[center_i-1:center_i+2, center_j-1:center_j+2] = True
quadratic_coefficients, _, rank, _ = np.linalg.lstsq(
    quadratic_design[fit_mask], relative_grid[fit_mask], rcond=None)
assert rank == 6
quadratic_prediction = quadratic_design @ quadratic_coefficients
quadratic_residual = relative_grid - quadratic_prediction
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].plot(bond_lengths, relative_grid[:, center_j], "o-", label="MMFF94 slice")
axes[0].plot(bond_lengths, quadratic_prediction[:, center_j], "--", label="Local quadratic fit")
axes[0].axvspan(bond_lengths[center_i-1], bond_lengths[center_i+1], alpha=0.15, color="gray", label="Fit region in this cut")
axes[0].set(xlabel="C–O length (Å)", ylabel="Relative energy (kcal/mol)", title="Fixed-angle cut")
axes[0].legend(fontsize=8)
limit = float(np.max(np.abs(quadratic_residual)))
im = axes[1].pcolormesh(bond_lengths, bond_angles, quadratic_residual.T,
                        cmap="RdBu_r", vmin=-limit, vmax=limit, shading="nearest")
axes[1].scatter([r0], [theta0], color="black", marker="+", s=80)
axes[1].set(xlabel="C–O length (Å)", ylabel="C–C–O angle (degree)", title="MMFF94 minus local quadratic model")
fig.colorbar(im, ax=axes[1], label="Residual (kcal/mol)")
fig.savefig(output_dir / "local_quadratic_validity.png", dpi=140)
plt.show()
print(f"Fit-region RMS residual: {np.sqrt(np.mean(quadratic_residual[fit_mask]**2)):.5f} kcal/mol")
print(f"Outside-fit RMS residual: {np.sqrt(np.mean(quadratic_residual[~fit_mask]**2)):.5f} kcal/mol")

**Conclusion.** A local quadratic description can closely reproduce nearby points while becoming less accurate away from them. Inspect both the size and the pattern of the residuals; the sign tells whether the local approximation overestimates or underestimates the actual slice energy.

**Next step:** choose a displacement range suited to the question, or use an anharmonic treatment when large motions matter. This fit is a two-coordinate, unrelaxed MMFF94 slice. It is neither the full mass-weighted Hessian nor a finite-temperature free-energy surface. The residuals measure approximation error against this same force field, not error against experiment.

### 8.4.6. What changes when some coordinates relax?

For specified scan coordinates $q=(r,\theta)$ and remaining independent coordinates $\mathbf x$, an ideal fully relaxed surface is

$$
E_\mathrm{relaxed}(q)=\min_{\mathbf x} E(q,\mathbf x).
$$

Its energy cannot exceed that of any particular allowed unrelaxed geometry at the **same** $q$, provided the same energy function is used and the minimization succeeds. In practice, local optimizers may find different conformational branches; a relaxed scan is not automatically a global search.

We will perform a smaller, precisely defined **partial relaxation** at seven bond lengths, all at $\theta_0$: fix the three heavy-atom Cartesian positions and optimize only the six hydrogens. This holds the selected length and angle exactly, and also keeps the C–C distance fixed. It is therefore an H-relaxed cut, not the fully relaxed surface defined above.

`AddFixedPoint` fixes coordinates directly. No artificial restraint energy is added. We still recompute the ordinary MMFF94 energy after optimization and check both convergence and the fixed coordinates.

In [ ]:
relax_indices = np.arange(0, len(bond_lengths), 2)  # seven grid points
relaxation_rows = []
start = perf_counter()
for i in relax_indices:
    molecule = scan_geometry(bond_lengths[i], theta0)
    heavy_before = molecule.GetConformer().GetPositions()[:3].copy()
    force_field = mmff_force_field(molecule)
    for atom_index in (0, 1, 2):
        force_field.AddFixedPoint(atom_index)
    force_field.Initialize()
    status = force_field.Minimize(maxIts=500, forceTol=1e-5, energyTol=1e-9)
    if status != 0:
        raise RuntimeError(f"H-only relaxation failed at grid index {i}: status {status}")
    np.testing.assert_allclose(molecule.GetConformer().GetPositions()[:3], heavy_before, atol=1e-12)
    np.testing.assert_allclose(rdMolTransforms.GetBondLength(molecule.GetConformer(), 1, 2),
                               bond_lengths[i], atol=1e-10)
    np.testing.assert_allclose(rdMolTransforms.GetAngleDeg(molecule.GetConformer(), 0, 1, 2),
                               theta0, atol=1e-9)
    relaxed_energy = mmff_force_field(molecule).CalcEnergy()
    unrelaxed_energy = energy_grid[i, center_j]
    if relaxed_energy > unrelaxed_energy + 1e-6:
        raise RuntimeError("Relaxation raised the energy beyond numerical tolerance.")
    relaxation_rows.append({"C–O length / Å": bond_lengths[i], "angle / degree": theta0,
                            "unrelaxed ΔE / kcal mol−1": unrelaxed_energy - reference_energy,
                            "H-relaxed ΔE / kcal mol−1": relaxed_energy - reference_energy,
                            "status": status})
relaxation_table = pd.DataFrame(relaxation_rows)
print(f"Seven H-only optimizations: {perf_counter() - start:.3f} s")
relaxation_table.round(6)

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4), layout="constrained")
ax.plot(bond_lengths[relax_indices], relative_grid[relax_indices, center_j], "o-", label="Unrelaxed deformation", markersize=4)
ax.plot(relaxation_table.iloc[:, 0], relaxation_table["H-relaxed ΔE / kcal mol−1"],
        "s--", label="Hydrogens relaxed; heavy atoms fixed", markersize=5)
ax.set(xlabel="C–O length (Å)", ylabel="ΔE relative to the same reference (kcal/mol)",
       title=f"Effect of partial relaxation at θ = {theta0:.2f}°")
ax.legend()
plt.show()
np.testing.assert_allclose(reference.GetConformer().GetPositions(), reference_xyz, atol=1e-12)

### 8.4.7. Energy, stationary points, and free energy

- **Grid minimum:** lowest evaluated energy among the sampled points. It depends on the coordinate window, spacing, and treatment of the other coordinates.
- **Local minimum:** stationary geometry with positive curvature in all nontrivial internal directions. A gradient and Hessian/frequency analysis are needed to classify a candidate; a contour map alone is insufficient.
- **First-order saddle point:** stationary geometry with one negative curvature direction. A ridge or maximum in a selected one-dimensional cut need not be a transition state. Reaction-path connectivity must also be assessed.
- **Free-energy profile:** describes a statistical ensemble at a specified temperature and with a defined coordinate measure. For example, $A(q)=-k_\mathrm BT\ln p(q)+C$ relates a coordinate probability density to a free-energy profile (use $R$, rather than $k_\mathrm B$, for molar energies). The measure/Jacobian matters when changing coordinates.

An unrelaxed potential-energy scan samples one geometry at each point. Minimization chooses an energetically favorable geometry. Neither operation sums over thermal configurations or supplies conformational entropy. Thus neither figure is a free-energy surface, an equilibrium population, or a reaction rate. Normalizing `exp(-energy_grid / RT)` would only assign weights to these selected grid geometries, not recover ethanol's full configurational ensemble.

The present MMFF94 graph also keeps all bonds intact. Large C–O stretching would probe the force field outside the intended local deformation example; it would not establish a dissociation energy or a reaction mechanism.

In [ ]:
# Save plain numeric results with method, axes, and the common reference energy.
scan_table = pd.DataFrame({"C_O_length_A": R.ravel(), "C_C_O_angle_deg": Theta.ravel(),
                           "MMFF94_energy_kcal_mol": energy_grid.ravel(),
                           "relative_energy_kcal_mol": relative_grid.ravel()})
scan_table.to_csv(output_dir / "ethanol_unrelaxed_scan.csv", index=False)
relaxation_table.to_csv(output_dir / "ethanol_h_relaxed_cut.csv", index=False)
(output_dir / "scan_definition.txt").write_text(
    f"Method: RDKit MMFF94; RDKit version: {rdBase.rdkitVersion}\n"
    "Atoms: C0-C1-O2; all six hydrogens explicit.\n"
    "Each point copies one MMFF94-optimized reference; set C1-O2 distance then C0-C1-O2 angle.\n"
    "The O-H group moves with O; no subsequent optimization in the 2D grid.\n"
    "The H-relaxed cut fixes heavy-atom Cartesian positions and minimizes hydrogen positions.\n"
    f"Common reference energy: {reference_energy:.12f} kcal/mol\n"
    f"Reference C-O length: {r0:.12f} angstrom; C-C-O angle: {theta0:.12f} degree\n",
    encoding="utf-8")
print(f"Saved numeric scans and their definition in {output_dir}.")

### Exercises

1. Why is creating a new randomly embedded conformer at every point an uncontrolled way to scan two coordinates?
2. Locate the lowest sampled point. Which further evidence would you need before calling it a minimum of the full potential energy surface?
3. Why do the H-relaxed points lie at or below the corresponding unrelaxed points? Why does this not establish a globally relaxed surface?
4. In the first slice plot, explain what `relative_grid[i, :]` fixes. What would `relative_grid[:, j]` fix?
5. Would a negative MMFF94 energy mean that ethanol is more stable than a molecule whose HF energy is −100 hartree? Explain the problem with that comparison.
6. Could this grid predict a C–O dissociation energy, an absorption wavelength, or a room-temperature conformer population? State what kind of additional model or calculation each quantity needs.
7. As a small numerical experiment, change the grid from 13 to 15 points per axis, keeping the same range and the center point. Does finer sampling alone improve the underlying force field's accuracy?

<details>
<summary>Suggested answers</summary>

1. Other coordinates and possibly conformational basins would change too. Energy differences would no longer isolate the stated deformation.
2. Optimize all relevant free coordinates, verify a small gradient, and inspect the nontrivial Hessian modes. A converged local minimum still does not prove global minimality.
3. The starting configuration is allowed in the H-only optimization, and the same MMFF94 energy is minimized. Only hydrogens are free; the C–C distance and selected heavy-atom geometry remain fixed, and local optimization need not explore every conformational branch.
4. `i` fixes C–O length and scans angles; `j` fixes the angle and scans lengths. Axis labels and array order must agree.
5. Different energy models use different reference zeros and describe different systems. Compare properly defined energy differences within a consistent model; unlike molecular formulas also require a balanced reaction or another explicitly defined comparison.
6. Dissociation needs a model capable of representing the products and changing bonding; absorption needs an excited-state treatment and transition information; populations need free energies and adequate sampling at a defined temperature and environment.
7. It improves resolution of the sampled model surface and may better locate its features. It does not repair force-field approximations or the omitted relaxations.

</details>

### References and next step

- [IUPAC: potential-energy surface](https://goldbook.iupac.org/terms/view/P04780) — meaning of a surface and selected coordinates.
- [Halgren, MMFF94: basis, form, scope, parameterization, and performance](https://doi.org/10.1002/%28SICI%291096-987X%28199604%2917%3A5%2F6%3C490%3A%3AAID-JCC1%3E3.0.CO%3B2-P) — primary description of the force field.
- [RDKit molecular transforms](https://www.rdkit.org/docs/source/rdkit.Chem.rdMolTransforms.html) — bond/angle units and which connected atoms move.
- [RDKit force-field helpers](https://www.rdkit.org/docs/source/rdkit.Chem.rdForceFieldHelpers.html) and [ForceField API](https://www.rdkit.org/docs/source/rdkit.ForceField.rdForceField.html) — parameter assignment, fixed points, and optimizer status.
- [Wong and York, *Exact Relation between Potential of Mean Force and Free-Energy Profile*](https://pmc.ncbi.nlm.nih.gov/articles/PMC3505112/) — statistical definitions and the importance of coordinate measure.

Next: [Part 8.5 — Vibrational analysis](Chapter08_Part5.ipynb).